In [0]:
# path Edu
recarga = spark.read.parquet("/Volumes/workspace/hackathon_2025/default/source/bases_recarga/BI_FP_ASS_RECARGA_CMV_NOVA/")

In [0]:
# path Michael
recarga = spark.read.parquet("/Volumes/hackathon_2025/default/source/bases_recarga/BI_FP_ASS_RECARGA_CMV_NOVA/")

In [0]:
display(recarga)

In [0]:
recarga.createOrReplaceTempView("recarga")

#Select Distinct

In [0]:
%sql
SELECT DISTINCT COD_TECNOLOGIA_DW FROM recarga

In [0]:
%sql
SELECT DISTINCT COD_CANAL_AQUISICAO FROM recarga

In [0]:
%sql
SELECT DISTINCT COD_TIPO_CREDITO FROM recarga

In [0]:
%sql
SELECT DISTINCT COD_PROMOCAO FROM recarga

In [0]:
%sql
SELECT DISTINCT COD_PLATAFORMA_ATU FROM recarga

In [0]:
%sql
SELECT DISTINCT COD_STATUS_PLATAFORMA FROM recarga

In [0]:
%sql
SELECT DISTINCT IND_METODO_PAGAMENTO FROM recarga

In [0]:
%sql
SELECT DISTINCT DW_PLANO_TARIFACAO FROM recarga

# 20160113 Michael

In [0]:
%sql
-- Volumetria e datas
SELECT
  COUNT(*) AS total_linhas,
  MIN(DAT_INSERCAO_CREDITO) AS min_dat_insercao_credito,
  MAX(DAT_INSERCAO_CREDITO) AS max_dat_insercao_credito
FROM recarga;

In [0]:
%sql
-- teste de Parsing de data (e taxa de inválidos)
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN DAT_INSERCAO_CREDITO IS NULL OR TRIM(DAT_INSERCAO_CREDITO)='' THEN 1 ELSE 0 END) AS null_dat,
  SUM(CASE WHEN DAT_INSERCAO_CREDITO IS NOT NULL AND TRIM(DAT_INSERCAO_CREDITO)<>'' 
            AND to_timestamp(DAT_INSERCAO_CREDITO,'ddMMMyyyy:HH:mm:ss') IS NULL
      THEN 1 ELSE 0 END) AS invalid_dat
FROM recarga;

In [0]:
%sql
-- “Chave” do evento + duplicidade (controle de dedupe)
-- Avaliando duplicidade por CPF + data+hora + valor

SELECT
  COUNT(*) AS total_linhas,
  COUNT(DISTINCT CONCAT(NUM_CPF,'#',DAT_INSERCAO_CREDITO,'#',HOR_INSERCAO_CREDITO,'#',VAL_REAL,'#',VAL_CREDITO_INSERIDO,'#',VAL_BONUS)) AS chaves_distintas,
  COUNT(*) - COUNT(DISTINCT CONCAT(NUM_CPF,'#',DAT_INSERCAO_CREDITO,'#',HOR_INSERCAO_CREDITO,'#',VAL_REAL,'#',VAL_CREDITO_INSERIDO,'#',VAL_BONUS)) AS possiveis_duplicadas
FROM recarga;

In [0]:
%sql
SELECT
  NUM_CPF, DAT_INSERCAO_CREDITO, HOR_INSERCAO_CREDITO,
  VAL_REAL, VAL_CREDITO_INSERIDO, VAL_BONUS,
  COUNT(*) AS qtd
FROM recarga
GROUP BY ALL
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

In [0]:
%sql
-- Sentinelas 
SELECT
  SUM(CASE WHEN COD_CANAL_AQUISICAO IN ('-1','-2','-3') THEN 1 ELSE 0 END) AS s_cod_canal_sentinela,
  SUM(CASE WHEN DW_TIPO_RECARGA IN ('-1','-2','-3') THEN 1 ELSE 0 END) AS s_tipo_recarga_sentinela,
  SUM(CASE WHEN DW_FORMA_PAGAMENTO IN ('-1','-2','-3') THEN 1 ELSE 0 END) AS s_forma_pag_sentinela,
  SUM(CASE WHEN DW_INSTITUICAO IN ('-1','-2','-3') THEN 1 ELSE 0 END) AS s_instit_sentinela,

  SUM(CASE WHEN CAST(VAL_BONUS AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS s_bonus_negativo,
  SUM(CASE WHEN CAST(VAL_REAL AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS s_real_negativo
FROM recarga;

In [0]:
%sql
-- SOS Recarga
SELECT
  FLAG_SOS,
  COUNT(*) AS qtd,
  MIN(CAST(VALOR_SOS AS DOUBLE)) AS min_valor_sos,
  MAX(CAST(VALOR_SOS AS DOUBLE)) AS max_valor_sos
FROM recarga
GROUP BY FLAG_SOS
ORDER BY FLAG_SOS;